# Quantum autoencoder for anomaly detection on LHC Olympics 2020

Runs the full study on a free Colab CPU: a trash-qubit quantum autoencoder
against four classical baselines, on the LHCO R&D dijet dataset.

**Runtime:** roughly 30-45 min for 5 seeds on a free CPU instance. No GPU needed
(the statevector simulation is 6 qubits = 64 amplitudes; the bottleneck is the
Python loop, not arithmetic).

**Data:** 74 MB high-level feature file from Zenodo record 6466204. The 2.9 GB
raw hadron-level file is *not* used.

Everything below calls into `src/`; the notebook is an entry point, not the
implementation. Read `src/qae.py` for the model and `src/evaluate.py` for the
metrics.

## 1. Environment

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO = "quantum-autoencoder-hep-anomaly-detection"

if IN_COLAB:
    if os.path.isdir(REPO):
        !cd {REPO} && git pull -q          # pick up any newer commits
    else:
        !git clone -q https://github.com/HaronJadid/{REPO}.git
    %cd {REPO}
    !pip install -q pennylane qiskit pylatexenc tables

sys.path.insert(0, os.getcwd())

import numpy, torch, pennylane, qiskit, sklearn
for m in (numpy, torch, pennylane, qiskit, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")

## 2. The circuit, and proof that the fast path is faithful

Training uses PennyLane's `default.qubit` with `diff_method="backprop"`, because
Qiskit's parameter-shift gradients need 2 circuit evaluations per parameter per
step (~10^6 executions per epoch here) which is not viable on a free CPU.

The Qiskit circuit remains the specification. The cell below asserts that the
two implementations produce the *same statevector* to machine precision, so the
speed-up costs nothing in fidelity to the stated architecture.

In [ ]:
from src.qae import build_qiskit_circuit, verify_against_qiskit, n_params

N_QUBITS, N_TRASH = 6, 2

# Ansatz depth is not fixed a priori: src/run_study.py selects it on
# background validation loss (no labels, no signal, no test data).
# Shown here at reps=3 purely to illustrate the circuit structure.
for reps in (1, 3, 5):
    for fm in ("ry", "zz"):
        d = verify_against_qiskit(N_QUBITS, reps, n_trials=5, feature_map=fm)
        print(f"{fm} encoding, reps={reps} ({n_params(N_QUBITS, reps):2d} par.): "
              f"max |Qiskit - PennyLane| = {d:.2e}")

qc, _, _ = build_qiskit_circuit(N_QUBITS, 3, feature_map="ry")
qc.decompose().draw(output="mpl", style="clifford", fold=40)

## 3. Data

Downloads and md5-verifies the LHCO R&D feature file, then builds the six
physics inputs. `mjj` is computed but deliberately **excluded** from the model
inputs: it is the resonance variable a real search bump-hunts in, so it is kept
aside purely as a sculpting diagnostic.

*If this cell fails with a 504, Zenodo is down -- it is intermittently
unavailable. Re-run it later; the download resumes from scratch but is only
74 MB.*

In [ ]:
from src.data import load_rnd, make_splits, FEATURES

df = load_rnd("data")
print("\nfeatures used as model input:", FEATURES)
print(f"background events: {(df.label==0).sum():,}")
print(f"signal events    : {(df.label==1).sum():,}")
df.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.ravel(), FEATURES + ["mjj"]):
    b, s = df[df.label == 0][col], df[df.label == 1][col]
    lo, hi = np.percentile(df[col], [0.5, 99.5])
    bins = np.linspace(lo, hi, 60)
    ax.hist(b, bins=bins, density=True, histtype="step", lw=1.6, label="background")
    ax.hist(s, bins=bins, density=True, histtype="step", lw=1.6, label="signal")
    ax.set_title(col + ("  (held out)" if col == "mjj" else ""), fontsize=9)
    ax.set_yscale("log")
axes.ravel()[0].legend(fontsize=8)
axes.ravel()[-1].axis("off")
fig.tight_layout()

## 3b. Is the encoded data compressible at all?

A trash-qubit autoencoder applies one fixed unitary and asks the trash
register to read |0...0>. By Ky Fan's theorem the best any ansatz can do is
the sum of the k largest eigenvalues of rho = E[|psi><psi|], with
k = 2^n_latent. That is an upper bound no depth or training budget can beat.

`ZZFeatureMap` applies a Hadamard layer and then only phase gates, so every
amplitude has the same modulus for every input and rho sits close to
maximally mixed. The bound then lands near the random-guess value.

In [ ]:
from src.encoding_analysis import compressibility, report
from src.qae import FEATURE_MAPS
from src.run_study import scale
from src.data import make_splits

sp = make_splits(df, n_train=25_000, n_val=5_000, seed=0)
xs, = scale(sp.train, seed=0)
for name, fm in FEATURE_MAPS.items():
    print(report(compressibility(fm, xs, N_QUBITS, N_TRASH), f"{name} encoding"))
    print()

## 4. Run the study

Five seeds. Each seed re-draws the train/val/test split *and* re-initialises
every model, so the quoted spread covers data variation as well as
initialisation. Within a seed, every model sees exactly the same events and is
trained by the same loop (`src/train.py`).

Set `--quick` for a fast pipeline check instead.

In [ ]:
!python -m src.run_study --seeds 0 1 2 3 4

## 5. Results

In [ ]:
import json
import pandas as pd

res = json.load(open("results/metrics.json"))
rows = []
for name, agg in res["summary"].items():
    rows.append({
        "model": name,
        "params": res["parameter_counts"][name],
        "AUC": f"{agg['auc']['mean']:.4f} ± {agg['auc']['std']:.4f}",
        "1/eps_B @ eps_S=0.3": f"{agg['rejection[eps_s=0.3]']['mean']:.1f} "
                               f"± {agg['rejection[eps_s=0.3]']['std']:.1f}",
        "rho(score, mjj)": f"{agg['spearman_score_vs_mjj']['mean']:+.3f}",
    })
pd.DataFrame(rows).set_index("model")

In [ ]:
from src.figures import make_all
from IPython.display import Image, display

# run_study already writes these; make_all() regenerates the ones that
# depend only on metrics.json.
make_all()
for f in ["roc.png", "auc.png", "scores.png", "sculpting.png",
          "training_curves.png", "circuit.png"]:
    path = f"results/figures/{f}"
    if os.path.exists(path):
        print(f)
        display(Image(path))